In [ ]:
import os
from pathlib import Path

import yaml
from natsort import natsorted

from pynxtools_apm.parsers.hfive_base import HdfFiveBaseParser

print(os.getcwd())
with open(os.path.join("examples", "source_directory.txt")) as fp:
    src_directory: str = f"{fp.read().strip().replace('/', os.sep)}"
print(src_directory)
trg_directory = os.path.join(src_directory, "..", "sha256")
print(trg_directory)

In [ ]:
for nexus_file_path in natsorted(
    Path(src_directory).glob("*.nxs"), key=lambda p: p.name
):  # sorting like e.g. nautilus explorer
    print(nexus_file_path)

    hfive_parser = HdfFiveBaseParser(
        file_path=nexus_file_path,
        hashing=True,
        verbose=False,
    )
    hfive_parser.get_content()
    hfive_parser.store_hashes(
        blacklist_by_key=[],
        blacklist_by_suffix=[],
        file_path=os.path.join(trg_directory, f"{nexus_file_path}.sha256.yml"),
    )

    del hfive_parser

In [ ]:
stats: dict[str, int] = {}
for yaml_file_path in natsorted(
    Path(trg_directory).glob("*.yml"), key=lambda p: p.name
):
    print(yaml_file_path)
    with open(yaml_file_path, encoding="utf-8") as fp:
        key_info: dict[str, str] = yaml.safe_load(fp)
        for key, value in key_info.items():
            if value.startswith("dst__0"):
                if key in stats:
                    stats[key] += 1
                else:
                    stats[key] = 1
        del key_info
print("Statistics collected")

In [ ]:
print(f"{len(stats)}\t\tunique named instances found in the set")
count = sum(
    [
        1 if "peak_identification/ion" not in key and "@" not in key else 0
        for key in sorted(stats, key=lambda x: x[1], reverse=True)
    ]
)
print(f"{count}\t\tnon-ion and non-attribute instances remain")
for key, count in sorted(stats.items(), key=lambda x: x[1], reverse=True):
    if "peak_identification/ion" not in key and "@" not in key:
        print(f"{count}\t\t{key}")